In [ ]:
# =========================================================
# LIBRARIES
# =========================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from prophet import Prophet
import matplotlib.pyplot as plt

### 📂 LOAD DATA
In this stage, data is loaded from an Excel file containing detailed information about electrical generation plants in Colombia.
The main columns in the file are:


⚙️ Type/Energy source


🔌 FUENTE DE ENERGIA


🏭 Dispatch


⚡ Generator


📈 Net effective capacity [MW]


🧑‍💼 Operator


🗓️ Commissioning date (FPO)


📍 Municipality


🗺️ Department


🌎 Sub-area


This data will serve as the basis for analyzing the evolution of installed capacity and trends by energy source type.

In [3]:
df_0 = pd.read_excel('../data/raw/PARATEC_Capacidadefectiva_27-10-2025.xlsx', sheet_name='CapacidadEfectiva_2', skiprows=7)

## 🧹 DATA CLEANING

Before performing any analysis, it is essential to ensure that the data is clean, consistent, and free of errors.
In this stage, the following tasks are carried out:

🧾 Data type correction, ensuring dates are in datetime format and capacities are numeric.

⚙️ Column name normalization.

This cleaning ensures that the database faithfully reflects the information from the generation plants, enabling accurate analyses and reliable results.

In [ ]:
# 1. Remove rows with no data

ultimas_5 = df_0.columns[-5:]  # Select the last 5 columns
df = df_0.dropna(subset=ultimas_5, how='all')

# 2. Rename column to a more descriptive name
df = df.rename(columns={'Tipo/Fuente de energía': 'NombrePlanta'})

# 3. Convert string data to numeric
df['Capacidad efectiva neta [MW]'] = (
    df['Capacidad efectiva neta [MW]']
    .astype(str)
    .str.replace('.', '', regex=False)   # Remove thousands separator
    .str.replace(',', '.', regex=False)  # Change decimal separator
    .astype(float)
)

# 4. Convert text to date with day/month/year format
df['Fecha de puesta en operación FPO'] = pd.to_datetime(df['Fecha de puesta en operación FPO'], format='%d/%m/%Y', errors='coerce')

### ⚙️ CALCULATE CUMULATIVE CAPACITY
Each record represents a plant with its respective net effective capacity.
To understand the evolution of the electrical system over time, the cumulative capacity is calculated — that is, the progressive sum of the capacities of all plants as they come into operation.
This value will allow us to visualize the historical growth of the generation infrastructure.

In [ ]:
df_series = df[['Fecha de puesta en operación FPO', 'Capacidad efectiva neta [MW]', 'FUENTE DE ENERGIA']].copy()

df_series = df_series.sort_values(by='Fecha de puesta en operación FPO', ascending=True).reset_index(drop=True)

# Create cumulative capacity column
df_series['Capacidad acumulada [MW]'] = df_series['Capacidad efectiva neta [MW]'].cumsum()

# ==============================
# 1. FIGURE: df_series
# ==============================
plt.figure(figsize=(5,3))
plt.plot(df_series['Fecha de puesta en operación FPO'],
         df_series['Capacidad acumulada [MW]'])
plt.xlabel('Date')
plt.ylabel('Cumulative capacity [MW]')
plt.title('Cumulative capacity growth (df_series)')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()

### 📊 Figure: cumulative capacity by generation type
This figure shows the evolution of cumulative capacity over time, differentiated by energy type or source (solar, hydraulic, thermal, etc.).
This chart helps identify which technologies have contributed most to the growth of the Colombian electrical system.

In [ ]:
plt.figure(figsize=(10,5))

# Iterate through each energy source category and plot with a label
for source, df_sub in df_series.groupby('FUENTE DE ENERGIA'):
    plt.scatter(df_sub['Fecha de puesta en operación FPO'],
                df_sub['Capacidad acumulada [MW]'],
                label=source)

plt.xlabel('Commissioning date (FPO)')
plt.ylabel('Cumulative capacity [MW]')
plt.title('Cumulative capacity by energy source type')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':')
plt.legend(title='Energy source')
plt.tight_layout()
plt.show()

### 📈 Figure: discretized and cumulative capacity by generation type
Here we present the comparison between:


Discretized capacity, which shows point increments when new plants come into operation.


Cumulative capacity, which reflects the total growth achieved up to each date.


This visualization helps understand the dynamics of new plant incorporation and their impact on the cumulative total.

In [ ]:
# Sort by date to ensure correct accumulation
df_cumul_source = df_series.sort_values(by='Fecha de puesta en operación FPO').copy()

# Cumulative by each energy source
df_cumul_source['Capacidad acumulada por fuente [MW]'] = (
    df_cumul_source
    .groupby('FUENTE DE ENERGIA')['Capacidad efectiva neta [MW]']
    .cumsum())

plt.figure(figsize=(7,3))

# Iterate through each energy source and plot its cumulative curve
for source, df_sub in df_cumul_source.groupby('FUENTE DE ENERGIA'):
    plt.plot(df_sub['Fecha de puesta en operación FPO'],
             df_sub['Capacidad acumulada por fuente [MW]'],
             label=source)

plt.xlabel('Commissioning date')
plt.ylabel('Cumulative capacity\nby source [MW]')
plt.title('Cumulative capacity growth by energy source')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':')
plt.legend(title='Energy source')
plt.tight_layout()
plt.show()

### 🧮 CREATE A DAILY DATAFRAME (DATE - VALUE)

To analyze growth continuously, it is necessary to build a daily time series.
The commissioning dates do not follow regular intervals, so a daily dataframe is generated between the minimum and maximum dates of the series.

The rule applied is:

If no new plant comes into operation on a given date, the cumulative capacity value remains the same as the previous day.

This way, a uniform series is obtained where the value remains constant between new entry dates, reflecting the actual behavior of cumulative growth.

In [ ]:
# 1. Keep only necessary columns
df_ts = df_series[['Fecha de puesta en operación FPO', 'Capacidad acumulada [MW]', 'FUENTE DE ENERGIA']].copy()

# 2. Group by date keeping the maximum value if there are duplicates
df_ts = df_ts.groupby('Fecha de puesta en operación FPO').max()

# 3. Create date index in ascending order
df_ts = df_ts.sort_index()

# 4. Create a daily range from the minimum to maximum date
date_range = pd.date_range(start=df_ts.index.min(), end=df_ts.index.max(), freq='D')

# 5. Reindex to the daily range and fill missing values with the last known value
df_ts = df_ts.reindex(date_range).ffill()

# 6. Rename index to Fecha
df_ts.index.name = 'Fecha'

# ==============================
# 2. FIGURE: df_ts (daily series)
# ==============================
plt.figure(figsize=(5,3))
plt.plot(df_ts.index,
         df_ts['Capacidad acumulada [MW]'])
plt.xlabel('Date')
plt.ylabel('Cumulative capacity [MW]')
plt.title('Daily cumulative capacity growth (df_ts)')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()

### 🔍 Comparison between the variable-interval series and the daily series

In this section, two representations of cumulative capacity are compared:

Variable-interval series: uses the actual commissioning dates.

Daily series: interpolates constant values for each day in the analyzed range.

This comparison evaluates the influence of temporal interpolation and analyzes the smoothness, continuity, and accuracy of both representations.

In [ ]:
# ==============================
# 3. FIGURE: Combined comparison
# ==============================
plt.figure(figsize=(6,4))
plt.plot(df_series['Fecha de puesta en operación FPO'],
         df_series['Capacidad acumulada [MW]'],
         label='Original data')
plt.plot(df_ts.index,
         df_ts['Capacidad acumulada [MW]'],
         label='Daily series')
plt.xlabel('Date')
plt.ylabel('Cumulative capacity [MW]')
plt.title('Cumulative capacity curve comparison')
plt.xticks(rotation=45)
plt.grid(True, linestyle=':')
plt.legend()
plt.tight_layout()
plt.show()

## ☀️ SOLAR ENERGY CAPACITY PROJECTION

In this phase, a projection of the growth of installed solar energy capacity is performed.
In Colombia, the trend points toward limiting the expansion of hydroelectric energy due to its high environmental costs, while actively promoting solar energy as a sustainable and rapidly growing alternative.

The projection is developed using four prediction models:

📉 Linear Regression

🌳 Random Forest

⚙️ XGBoost

🔮 Prophet

To improve model performance and stability, a 30-day rolling mean is applied to the cumulative solar capacity series.
This smoothing reduces short-term fluctuations and allows for clearer capture of the overall solar growth trend.

In [ ]:
# ============================================================
# 1. VARIABLES
# ============================================================
# --- 1. Filter only solar plants ---
df_ts_solar = df_ts[df_ts['FUENTE DE ENERGIA'] == 'PLANTA SOLAR'].copy()

# --- 2. Sort index by date ---
df_ts_solar = df_ts_solar.sort_index()

# --- 3. Create daily range from minimum to maximum date ---
date_range = pd.date_range(start=df_ts_solar.index.min(), end=df_ts_solar.index.max(), freq='D')

# --- 4. Reindex to include all dates in the range ---
df_ts_solar = df_ts_solar.reindex(date_range)

# --- 5. Fill missing values forward and backward ---
df_ts_solar = df_ts_solar.ffill().bfill()

# --- 6. Rename the index ---
df_ts_solar.index.name = 'Fecha'

# Create continuous time variable (days since initial date)
df_ts_solar['t'] = (df_ts_solar.index - df_ts_solar.index.min()).days

# df_ts_solar['Capacidad_suavizada'] = df_ts_solar['Capacidad acumulada [MW]'].ewm(span=10, adjust=False).mean()
df_ts_solar['Capacidad_suavizada'] = df_ts_solar['Capacidad acumulada [MW]'].rolling(window=30, min_periods=1).mean()

# Dependent variable (target)
y = df_ts_solar['Capacidad_suavizada']

# Independent variables
X = pd.DataFrame({'t': df_ts_solar['t']})

# ============================================================
# 2. MODELS
# ============================================================

models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(
        n_estimators=800,
        max_depth=4,
        min_samples_leaf=5,
        random_state=42
    ),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=42
    )
}

# Train classical models
for name, model in models.items():
    model.fit(X, y)

# ============================================================
# 3. PROPHET
# ============================================================

df_prophet = df_ts_solar.reset_index()[['Fecha', 'Capacidad_suavizada']].rename(
    columns={'Fecha': 'ds', 'Capacidad_suavizada': 'y'}
)

prophet_model = Prophet(yearly_seasonality=False, daily_seasonality=False)
prophet_model.fit(df_prophet)

# ============================================================
# 4. FUTURE PROJECTIONS
# ============================================================

max_date = df_ts_solar.index.max()
future_dates = pd.date_range(max_date + pd.Timedelta(days=1), '2030-12-31', freq='D')
t_fut = (future_dates - df_ts_solar.index.min()).days
X_fut = pd.DataFrame({'t': t_fut})

# Predictions
predictions = {name: model.predict(X_fut) for name, model in models.items()}

# Prophet
future_prophet = prophet_model.make_future_dataframe(periods=len(future_dates), freq='D')
forecast_prophet = prophet_model.predict(future_prophet)
pred_prophet = forecast_prophet.set_index('ds').loc[future_dates, 'yhat']

predictions['Prophet'] = pred_prophet.values

# ============================================================
# 5. MODEL EVALUATION (ON HISTORICAL DATA)
# ============================================================

metrics_list = []
for name, model in models.items():
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    rmse = mean_squared_error(y, y_pred) ** 0.5
    metrics_list.append([name, r2, mae, rmse])

# Prophet (evaluation on actual data)
y_pred_prophet = prophet_model.predict(df_prophet)[['yhat']].values.flatten()
r2 = r2_score(y, y_pred_prophet)
mae = mean_absolute_error(y, y_pred_prophet)
rmse = mean_squared_error(y, y_pred_prophet) ** 0.5
metrics_list.append(['Prophet', r2, mae, rmse])

df_metrics = pd.DataFrame(metrics_list, columns=['Modelo', 'R2', 'MAE', 'RMSE'])
print(df_metrics)

# ============================================================
# 6. UNIFY RESULTS FOR SAVING
# ============================================================

# Actual data
df_actual = df_ts_solar[['Capacidad_suavizada']].rename(columns={'Capacidad_suavizada': 'Histórico'})

# Future data
df_future = pd.DataFrame(index=future_dates)
for name, y_pred in predictions.items():
    df_future[name] = y_pred

# Concatenate
df_result = pd.concat([df_actual, df_future])

# Save to Excel
output_path = '../results/proyeccion_capacidad_solar.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_result.to_excel(writer, sheet_name='Proyeccion')
    df_metrics.to_excel(writer, sheet_name='Metricas', index=False)

print(f"Results saved to: {output_path}")

# ============================================================
# 7. PLOT
# ============================================================

plt.figure(figsize=(12, 6))
plt.plot(df_actual.index, df_actual['Histórico'], color='black', linewidth=2.2, label='Historical')

for name in predictions.keys():
    plt.plot(future_dates, df_future[name], '--', linewidth=2, label=name)

plt.axvline(df_actual.index.max(), color='gray', linestyle=':', label='End of actual data')
plt.xlabel('Date')
plt.ylabel('Cumulative capacity [MW]')
plt.title('Solar cumulative capacity projection through 2030')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()